In [2]:
!pip install pennylane


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 102.1 MB/s eta 0:00:00


In [3]:
import pennylane as qml
from pennylane import numpy as np

Modelo de Ising básico con PennyLane:implementar el Hamiltoniano de Ising más simple:2 espines (1D), interacción entre vecinos, campo magnético externo: H=−JZ0​Z1​−h(Z0​+Z1​)

Definimos el sistema cuántico (2 qubits)

In [5]:
dev = qml.device("default.qubit", wires=2) #no hardware real, 2 qubits

Definimos el Hamiltoniano de Ising

In [6]:
def ising_hamiltonian(J, h):
    coeffs = [-J, -h, -h]
    obs = [
        qml.PauliZ(0) @ qml.PauliZ(1),  # interacción spin-spin,mide si los spins están alineados,representa interacción ferromagnética
        qml.PauliZ(0),                  # campo externo en spin 0,fuerza que empuja los spins a alinearse en una dirección, representa “campo magnético externo”
        qml.PauliZ(1)                   # campo externo en spin 1
    ]
    return qml.Hamiltonian(coeffs, obs)

Circuito cuántico simple (estado base variacional)

In [11]:
@qml.qnode(dev)
def circuit(params, J, h):
    # estado inicial en superposición
    qml.Hadamard(wires=0)
    qml.Hadamard(wires=1)

    # capa variacional simple
    qml.RX(params[0], wires=0)#dependen de parámetros ajustables que se optimizan para minimizar energía
    qml.RX(params[1], wires=1)

    H = ising_hamiltonian(J, h) #calcula la energía esperada del sistema

    return qml.expval(H)#devuelve un número (coste)

Función de coste (energía del sistema)

In [12]:
def cost(params, J, h):
    return circuit(params, J, h)

Optimización (encontrar estado de mínima energía)

In [13]:
J = 1.0   # interacción
h = 0.5   # campo externo

params = np.random.randn(2, requires_grad=True)

opt = qml.GradientDescentOptimizer(stepsize=0.1)

for i in range(50):
    params = opt.step(lambda v: cost(v, J, h), params)

    if i % 10 == 0:
        print(f"Paso {i} | Energía = {cost(params, J, h):.4f}")

Paso 0 | Energía = 0.0000
Paso 10 | Energía = 0.0000
Paso 20 | Energía = 0.0000
Paso 30 | Energía = 0.0000
Paso 40 | Energía = 0.0000


circuito genera estado totalmente simétrico